In [ ]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [ ]:
doc = documents[0]
print(doc["filename"])
print(doc["content"])

In [ ]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
import json

for doc in documents:
    user_prompt = json.dumps({
        "filename": doc["filename"],
        "content": doc["content"]
    })

In [ ]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [ ]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [ ]:
result = response.output_parsed

print(result)

questions = response.output_parsed.questions

for question in questions:
    records.append({
        "filename": doc["filename"],
        "question": question
    })

In [ ]:
from evaluation_utils import llm_structured

In [ ]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

In [ ]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["filename"]
    })

records

In [ ]:
print(records[0])
print(records[1])
print(records[2])

In [ ]:
response.usage.input_tokens

In [ ]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")


In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

In [ ]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

In [ ]:
def text_search(query, num_results=5):
    boost_dict = {"content": 1.0}

    return index.search(
        query,
        num_results=num_results,
        boost_dict=boost_dict
    )

In [150]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank + 1)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
texts = [doc["content"] for doc in chunks]

X = model.encode(texts)

In [ ]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [ ]:
def vector_search(query, num_results=5):
    v_query = model.encode(query)

    return vindex.search(
        v_query,
        num_results=num_results
    )

In [151]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k, num_results = 5)

In [ ]:
q = ground_truth[0]["question"]
q

In [ ]:
hybrid_search(q)

In [ ]:
vector_search(q)

In [76]:
q = ground_truth[0]["question"]
expected_filename = ground_truth[0]["filename"]

results = text_search(query=q)

for d in results:
    print(f'{d["filename"]} == {expected_filename}: {d["filename"] == expected_filename}')

01-agentic-rag/lessons/03-rag.md == 01-agentic-rag/lessons/01-intro.md: False
01-agentic-rag/lessons/13-function-calling.md == 01-agentic-rag/lessons/01-intro.md: False
01-agentic-rag/lessons/03-rag.md == 01-agentic-rag/lessons/01-intro.md: False
01-agentic-rag/lessons/13-function-calling.md == 01-agentic-rag/lessons/01-intro.md: False
01-agentic-rag/lessons/01-intro.md == 01-agentic-rag/lessons/01-intro.md: True


In [104]:
relevance = []

for d in results:
    relevance.append(int(d["filename"] == expected_filename))

relevance

[0, 0, 0, 0, 1]

In [105]:
def compute_relevance_text(q):
    expected_filename = q["filename"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == expected_filename))

    return relevance

In [97]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [106]:
def compute_relevance(q, search_function):
    expected_filename = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == expected_filename))

    return relevance

In [107]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [123]:
def hit_rate(relevance_total):
    cnt = 0

    for line in relevance_total:
        if 1 in line:
            cnt += 1

    return cnt / len(relevance_total)

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [127]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [128]:
evaluate(ground_truth, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [129]:
evaluate(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.8083333333333333, 'mrr': 0.6356944444444446}

In [146]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [152]:
for k in [1, 50, 100, 200]:
    result = evaluate(ground_truth,
             lambda query, k=k: hybrid_search(query, k=k)
             )

    print(f"k={k}: {result}")

  0%|          | 0/360 [00:00<?, ?it/s]

k=1: {'hit_rate': 0.8611111111111112, 'mrr': 0.6771296296296299}


  0%|          | 0/360 [00:00<?, ?it/s]

k=50: {'hit_rate': 0.8472222222222222, 'mrr': 0.6721296296296295}


  0%|          | 0/360 [00:00<?, ?it/s]

k=100: {'hit_rate': 0.8472222222222222, 'mrr': 0.6721296296296295}


  0%|          | 0/360 [00:00<?, ?it/s]

k=200: {'hit_rate': 0.8472222222222222, 'mrr': 0.6721296296296295}
